<a href="https://colab.research.google.com/github/lannd3217/Interview_RAG/blob/main/Retrieval_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/lannd3217/Interview_RAG.git

In [ ]:
%cd Interview_RAG

In [ ]:
%%capture
!pip install -U langchain langchain-community langchain-openai
!pip install -U langchain langchain-community langchain-text-splitters
!pip install -U pymupdf langchain-community
!pip install -U langchain-huggingface sentence-transformers
!pip install -q sentence-transformers faiss-cpu transformers
!pip install langchain langchain-community langchain-chroma langchain-huggingface pymupdf sentence-transformers
!pip install "unstructured[all-docs]"


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Instantiate Retriever
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma(
    persist_directory="./interview_vector_db",
    embedding_function=embedding_model,
    collection_name = "interview_prep_collection"
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2} # Fetches top-2 most relevant chunks
)

In [ ]:
test_docs = retriever.invoke("technical interview")
print(f"Success! Found {len(test_docs)} relevant chunks.")
if len(test_docs) > 0:
    print(f"Snippet: {test_docs[0].page_content[:100]}...")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create Prompt Template
template = """
Use the following pieces of context to answer the question at the end.
If don't know the answer, say I don't know.
Context: {context}
Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template) #

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
import torch

# Initialize Free Local LLM (HuggingFace Flan-T5)
llm = HuggingFacePipeline.from_model_id(
    model_id="google/flan-t5-large",
    task="text2text-generation",
    device = 0 if torch.cuda.is_available() else -1,
    pipeline_kwargs={
        "max_new_tokens": 200,
        "truncation": True,
        "max_length": 512
        }
)

# Assemble the Chain
# chain = (
#     {"context": retriever, "question": RunnablePassthrough()} #
#     | prompt
#     | llm
#     | StrOutputParser() #
# )



# A helper function to extract ONLY the text content from retrieved documents
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Re-assemble the Chain
chain = (
    {
        "context": retriever | RunnableLambda(combine_docs), # Extracts only text
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
q = "What is the best way to handle a technical interview?"
a = chain.invoke(q)
print(a)

In [ ]:
test_questions = [
    "What is the best way to handle a technical interview?",
    "What do people on Reddit recommend doing if you get stuck on a coding question during a live interview?",
    "What are some common 'red flags' to look for in a company's data science culture according to interviewees?",
    "Based on the provided documents, how is the 'Bias-Variance Tradeoff' explained?",
    "What are the primary differences between supervised and unsupervised learning mentioned in the text?",
    "What are the three most common evaluation metrics for classification models according to the source?",
    "How do I explain my data science projects?",
    "What are the main mistakes candidates make in interviews?",
    "Explain the difference between L1 and L2 regularization and when to use each",
    "What steps should I take to transition from a data analyst to a data scientist, and what skills are most important?"
]


In [ ]:

print("Running Batch Test...\n")

for q in test_questions:
    print(f"Question: {q}")
    answer = chain.invoke(q)
    print(f"Answer: {answer}")
    print("-" * 40)

In [ ]:
## HYBRID RETRIEVERS

# 1. Fetch data from Chroma
raw_data = vector_store.get()

# 2. Reconstruct Document objects using the core import
from langchain_core.documents import Document

docs_from_db = [
    Document(
        page_content=text,
        metadata=meta
    )
    for text, meta in zip(raw_data["documents"], raw_data["metadatas"])
]

from langchain_classic.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# 1. Initialize BM25 using the docs we just pulled from your DB
bm25_retriever = BM25Retriever.from_documents(docs_from_db)
bm25_retriever.k = 3

# 2. Use your existing vector_store as the dense retriever
# dense_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 3. Create the Hybrid Retriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever],
    weights=[0.4, 0.6]
)

# 4. Run your query
# query = "How do I handle a technical interview for a data science role?"
# results = hybrid_retriever.invoke(query)

# # --- Display Results ---
# print(f"Found {len(results)} relevant chunks via Hybrid Search\n")
# for i, doc in enumerate(results):
#     print(f"-- Result {i+1} --")
#     print(f"Source: {doc.metadata.get('source', 'Unknown')}")
#     print(f"Snippet: {doc.page_content}...\n")

In [ ]:
# raw_data["documents"]

In [ ]:
## HYBRID RETRIEVER TESTING

hybrid_template = """
You are a professional Interview Coach. Use the provided Context to answer the Question.
If the context contains code snippets that do not explain the concept, ignore the code.
If the answer is not in the context, say "I'm sorry, I don't have that specific information in my documents."

Context:
{context}

Question:
{question}

Answer:
"""
hybrid_prompt = ChatPromptTemplate.from_template(template)

hybrid_chain = (
    {
        "context": hybrid_retriever | RunnableLambda(combine_docs), # Extracts only text
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)
# test_questions = [
#     "What is the best way to handle a technical interview?",
#     "What do people on Reddit recommend doing if you get stuck on a coding question during a live interview?",
#     "What are some common 'red flags' to look for in a company's data science culture according to interviewees?",
#     "Based on the provided documents, how is the 'Bias-Variance Tradeoff' explained?",
#     "What are the primary differences between supervised and unsupervised learning mentioned in the text?",
#     "What are the three most common evaluation metrics for classification models according to the source?",
#     "How do I explain my data science projects?",
#     "What are the main mistakes candidates make in interviews?",
#     "Explain the difference between L1 and L2 regularization and when to use each",
#     "What steps should I take to transition from a data analyst to a data scientist, and what skills are most important?"
# ]

print("Running Batch Test...\n")

for q in test_questions:
    print(f"Question: {q}")
    answer = hybrid_chain.invoke(q)
    print(f"Answer: {answer}")
    print("-" * 40)

In [ ]:
# 1. Utility function to print the list of ranked documents (From your screenshot)
def dump_doc_source(result_documents):
    for doc in result_documents:
        # We print the source and a snippet of content to see WHAT was retrieved
        source = doc.metadata.get("source", "Unknown")
        content_snippet = doc.page_content.replace("\n", " ")
        print(f"Source: {source} | Snippet: {content_snippet}...")
    print("\n")

# 2. Test input list
test_inputs = [
    "rag is cheaper",
    "benefits of rag",
    "piece of cloth",
    "What is the Bias-Variance Tradeoff?" # Added this to debug your previous error
]

# 3. Change input index for testing (ndx = 0, 1, 2, or 3)
ndx = 3
print(f"TESTING INPUT: '{test_questions[3]}'\n")
print("="*50)

# --- Dump the ranked list for BM25 (Keyword Search) ---
print(" BM25 (Keyword)")
print("-----")
results_bm25 = bm25_retriever.invoke(test_inputs[ndx])
dump_doc_source(results_bm25)

# --- Dump the ranked list for ChromaDB (Semantic Search) ---
print("ChromaDB (Vector/Dense)")
print("----------")
# Note: Ensure your retriever variable is named 'dense_retriever' or 'chromadb_retriever'
results_chromadb = retriever.invoke(test_inputs[ndx])
dump_doc_source(results_chromadb)

# --- Dump the final ranked list for Ensemble Retriever (Hybrid) ---
print("Ensemble Retriever (Hybrid)")
print("------------------")
results = hybrid_retriever.invoke(test_inputs[ndx])
dump_doc_source(results)

# Return the full objects for inspection if needed
# results

In [ ]:
## QUERY REWRITTING

rewrite_template = """
Original question: {question}

Rewrite the question to be more specific and detailed for searching a document database. Include relevant technical terms.
Rewritten question:
"""
rewrite_prompt = ChatPromptTemplate.from_template(rewrite_template)
query_rewriter = rewrite_prompt | llm | StrOutputParser()

# --- SETUP CHAIN ---
rewrite_template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
rewrite_prompt = ChatPromptTemplate.from_template(rewrite_template)



# --- EXECUTE ---
original_query = "What is the best way to handle a technical interview?"
print(f"Original: {original_query}")

# 1. Rewrite the query
rewritten_query = query_rewriter.invoke({"question": original_query})
print(f"Rewritten: {rewritten_query}")

# 2. Build and run the chain with the rewritten query
rewrite_chain = (
    {"context": hybrid_retriever | combine_docs, "question": RunnablePassthrough()}
    | rewrite_prompt
    | llm
    | StrOutputParser()
)

print("\nRunning Rewrite Chain...")

for q in test_questions:
    print(f"Question: {q}")
    print(f"Rewritten Question: {query_rewriter.invoke({"question": q})}")
    answer = rewrite_chain.invoke(q)
    print(f"Answer: {answer}")
    print("-" * 40)

In [ ]:
## QUERY DECOMPOSITION
decomposition_template = """
Break the user question into exactly two distinct sub-questions for database searching.
Do not repeat the user question.

User question: What are the common mistakes in project portfolios and how can I fix them?
Sub-questions:
What are common portfolio mistakes?
How to fix portfolio mistakes?

User question: {question}
Sub-questions:
"""
decomposition_prompt = ChatPromptTemplate.from_template(decomposition_template)
decomposer = decomposition_prompt | llm | StrOutputParser()

# --- SETUP CHAIN ---
decomposition_template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
decomposition_prompt = ChatPromptTemplate.from_template(template)


decomposition_chain = (
    # {"context": hybrid_retriever | combine_docs, "question": RunnablePassthrough()}

    decomposition_prompt
    | llm
    | StrOutputParser()
)

# # --- EXECUTE DECOMPOSITION ---
# original_query = "What are the common mistakes in project portfolios and how can I fix them?"
# print(f"Original: {original_query}")

# # 1. Decompose the query
# sub_questions_text = decomposer.invoke({"question": original_query})
# # Simple parsing to split sub-questions - ensure your LLM output matches this
# sub_questions = [sq.strip() for sq in sub_questions_text.split('\n') if sq.strip()]
# print(f"Sub-questions: {sub_questions}")

# # 2. Retrieve context for all sub-questions
# combined_docs = []
# for sq in sub_questions:
#     combined_docs.extend(hybrid_retriever.invoke(sq))

# # Remove duplicates based on page_content
# unique_docs = list({doc.page_content: doc for doc in combined_docs}.values())

# # 3. Format the docs for the LLM
# formatted_context = combine_docs(unique_docs)

# 4. Run the chain with the combined context directly
print("\nRunning RAG Chain with decomposed context...")
for q in test_questions:
    print(f"Question: {q}")
    sub_questions_text = decomposer.invoke({"question": q})
    sub_questions = [sq.strip() for sq in sub_questions_text.split('\n') if sq.strip()]
    print(f"Sub-questions: {sub_questions}")

    # 2. Retrieve context for all sub-questions
    combined_docs = []
    for sq in sub_questions:
        combined_docs.extend(hybrid_retriever.invoke(sq))

    # Remove duplicates based on page_content
    unique_docs = list({doc.page_content: doc for doc in combined_docs}.values())

    # 3. Format the docs for the current question
    formatted_context = combine_docs(unique_docs)

    # 4. Run the chain with the combined context for this specific question
    # Fix: pass 'q' (the current question in the loop)
    response = decomposition_chain.invoke({
        "context": formatted_context,
        "question": q
    })

    print(f"Answer: {response}")
    print("-" * 40)